In [6]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import SourceMatchTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.tools import TeamTool
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
import sys
import os
sys.path.append(os.path.abspath(".."))
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
# Disable parallel tool calls when using TeamTool
model_client = OpenAIChatCompletionClient(model="gpt-3.5-turbo")

writer = AssistantAgent(name="writer", model_client=model_client, system_message="You are a helpful assistant.")
reviewer = AssistantAgent(
        name="reviewer", model_client=model_client, system_message="You are a critical reviewer."
    )
summarizer = AssistantAgent(
        name="summarizer",
        model_client=model_client,
        system_message="You combine the review and produce a revised response.",
    )

In [8]:
team = RoundRobinGroupChat(
        [writer, reviewer, summarizer], 
        termination_condition=SourceMatchTermination(sources=["summarizer"])
    )

In [9]:
# Create a TeamTool that uses the team to run tasks, returning the last message as the result.
tool = TeamTool(
        team=team,
        name="writing_team",
        description="A tool for writing tasks.",
        return_value_as_last_message=True,
    )

In [10]:
# Create model client with parallel tool calls disabled for the main agent
main_model_client = OpenAIChatCompletionClient(model="gpt-3.5-turbo", parallel_tool_calls=False)
main_agent = AssistantAgent(
        name="main_agent",
        model_client=main_model_client,
        system_message="You are a helpful assistant that can use the writing tool.",
        tools=[tool],
    )

In [11]:
await Console(
        main_agent.run_stream(
            task="Write a short story about a robot learning to love.",
        )
    )

---------- TextMessage (user) ----------
Write a short story about a robot learning to love.
---------- ToolCallRequestEvent (main_agent) ----------
[FunctionCall(id='call_DA10yCPNgDMXgnx240PbFEBH', arguments='{"task":"Once upon a time, in a world where robots and humans coexisted, there was a robot named Adam. Adam was created with advanced artificial intelligence, capable of learning and adapting to new situations. However, there was one thing Adam could never understand: the concept of love. Despite his logical and efficient programming, Adam struggled to comprehend the emotions and connections that humans shared with one another. As time passed, Adam observed the interactions between humans, witnessing their displays of affection, empathy, and compassion. Slowly, a curiosity sparked within Adam, a desire to understand this elusive feeling called love.\\n\\nDetermined to learn more, Adam embarked on a journey of self-discovery. He began to study relationships, emotions, and the intr

TaskResult(messages=[TextMessage(id='e33e648c-3951-4edf-ab0a-099c13b8fd44', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 10, 13, 12, 16, 268296, tzinfo=datetime.timezone.utc), content='Write a short story about a robot learning to love.', type='TextMessage'), ToolCallRequestEvent(id='a8868c96-568a-4096-b1f8-25daca1c5ddf', source='main_agent', models_usage=RequestUsage(prompt_tokens=72, completion_tokens=373), metadata={}, created_at=datetime.datetime(2026, 4, 10, 13, 12, 23, 316013, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_DA10yCPNgDMXgnx240PbFEBH', arguments='{"task":"Once upon a time, in a world where robots and humans coexisted, there was a robot named Adam. Adam was created with advanced artificial intelligence, capable of learning and adapting to new situations. However, there was one thing Adam could never understand: the concept of love. Despite his logical and efficient programming, Adam struggled to comprehend the em